In [ ]:
import os, time, requests, jieba
from typing import List


# 引入 LlamaIndex 核心元件
from llama_index.core import StorageContext, Settings, load_index_from_storage, VectorStoreIndex
from llama_index.core.prompts.prompts import SimpleInputPrompt
from llama_index.core.schema import Document
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.node_parser import SentenceSplitter 

# 引入 LlamaIndex 的 Ollama 整合
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

import warnings
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
warnings.filterwarnings("ignore", category=RuntimeWarning, message="invalid value encountered in scalar divide")

class LlamaIndexRAG:
    """
    使用 LlamaIndex 的 VectorStoreIndex 實現 RAG 系統
    """
    def __init__(self, model_name="llama3:8b", embed_model_name="mxbai-embed-large", base_url="http://localhost:11434"):
        self.model_name = model_name
        self.embed_model_name = embed_model_name
        self.base_url = base_url
        self.index = None

        # 檢查 Ollama 服務是否運行
        self._check_ollama_status()

        # 配置 LlamaIndex 的 LLM 和嵌入模型
        self.llm = Ollama(model=self.model_name, base_url=self.base_url, request_timeout=30000,
                          additional_kwargs={
                            "num_ctx": 8192,       # prompt總長度限制，避免一次塞太多 token
                            "num_predict": 512     # 限制輸出長度，避免回應太大
                        })
        self.embed_model = OllamaEmbedding(model_name=self.embed_model_name, base_url=self.base_url,
                        request_timeout=6000, # 增加超時等待
                        additional_kwargs={
                            "num_ctx": 8192,  # 這裡加到 8192
                        })

        Settings.llm = self.llm
        Settings.embed_model = self.embed_model
        Settings.embed_batch_size = 1      # 避免 embedding batch 太大
        Settings.chunk_size =  512         # 每個 chunk 大小
        Settings.chunk_overlap = 0         # chunk 重疊 (招生文本不需要overlap)

        # 查詢模板
        self.qa_template = SimpleInputPrompt("""
        請參考以下資訊回答問題：
        ---------------------
        {context_str}
        ---------------------
        問題：{query_str}
        請以繁體中文回答，並根據所提供的資訊進行回答。如果資訊不涵蓋問題，請回答：我不知道。
        """)

    def _check_ollama_status(self):
        """檢查 Ollama 服務器是否可達"""
        try:
            response = requests.get(f"{self.base_url}/api/tags")
            if response.status_code == 200:
                print(f"✅ Ollama 服務器 {self.base_url} 可達。")
            else:
                raise ConnectionError(f"無法連接到 Ollama 服務器。狀態碼: {response.status_code}")
        except requests.exceptions.ConnectionError as e:
            raise ConnectionError(f"無法連接到 Ollama 服務器，請檢查服務是否已啟動: {e}")

    def create_index_from_documents(self, documents: List[str], index_dir: str = "storage"):
        try:
            storage_context = StorageContext.from_defaults(persist_dir=index_dir)
            self.index = load_index_from_storage(storage_context)
            print(f"✅ 成功載入現有索引於 '{index_dir}'。")
        except:
            print("📝 找不到現有索引或載入失敗，正在重新建立 VectorStoreIndex...")
            
            # 1. 將原始文本轉換為 Document 物件
            docs = [Document(text=d) for d in documents]
            
            # 2. 強制定義一個切片器 (Splitter)
            splitter = SentenceSplitter(chunk_size=4096, chunk_overlap=0)
            
            # 3. 執行切片，將 Documents 轉為 Nodes
            nodes = splitter.get_nodes_from_documents(docs)
            print(f"📄 原始文檔 {len(docs)} 頁，切片後產生 {len(nodes)} 個節點。")           
       
            # 4. 使用這些切好的 nodes 建立索引 (這一步最花時間，我們單獨幫它計時)
            print("⏳ 正在進行 Embedding 並建置 Vector Index (這可能需要一些時間)...")
            vector_start_time = time.time()
            
            self.index = VectorStoreIndex(nodes)
            
            vector_duration = time.time() - vector_start_time            
            # 儲存索引
            self.index.storage_context.persist(persist_dir=index_dir)
            print(f"✅ 已成功建立並儲存 VectorStoreIndex。")
            print(f"⏱️ 核心建置 Vector Index 耗時：{vector_duration:.2f} 秒。")
            
    def print_vector_index_structure(self, max_nodes: int = 20):
        """
        列印 VectorStoreIndex 的節點摘要資訊。
        Args: max_nodes: 最多列印的節點數，避免輸出過長
        """
        if self.index is None:
            print("❌ 錯誤：索引尚未建立。")
            return
        print("\n📦 VectorStoreIndex 節點資訊:\n","=" * 50)

        # 從 docstore 取出所有節點
        all_nodes = list(self.index.docstore.docs.values())
        if not all_nodes:
            print("❌ docstore 中沒有任何節點。")
            return
        print(f"共有 {len(all_nodes)} 個節點。")

        for i, node in enumerate(all_nodes[:max_nodes]):
            summary = node.get_content()[:80].replace("\n", " ") + "..."
            file_name = node.metadata.get('file_name', 'N/A')
            page_label = node.metadata.get('page_label', 'N/A')
            print("-" * 50, f"\n[{i+1}] Node ID: {node.node_id}")
            print(f"來源檔案: {file_name}, 頁碼: {page_label}\n",f"摘要: {summary}")

        if len(all_nodes) > max_nodes:
            print(f"... (僅顯示前 {max_nodes} 個節點)")
        print("=" * 50)

    def rag_query(self, query: str, topK=5) -> str:
        """
        對索引進行查詢並獲取 RAG 回答
        """
        if self.index is None:
            return "錯誤：索引尚未建立，無法進行查詢。"
        print("🔍 正在查詢...")
        query_engine = self.index.as_query_engine(
            text_qa_template=self.qa_template,
            similarity_top_k=topK
        )
        try:
            response = query_engine.query(query)
            print(f"🤖 回答: {response.response}\n")
            print("---\n","📄 來源文件 (Source Nodes):")
            if response.source_nodes:
                for i, node in enumerate(response.source_nodes):
                    node_content = node.get_content().strip()
                    print(f"[{i+1}] Node ID: {node.node_id}")
                    print(f"內容摘要: {node_content[:10]}...\n","-" * 20)
            else:
                print("❌ 找不到相關的來源節點。")
            return response.response
        except Exception as e:
            return f"❌ 查詢時發生錯誤：{e}"

# 使用範例
def main(data_file, idx_path, model_name="llama3:8b",jb: bool = False, emodel_name="mxbai-embed-large"):
    print(f"🤖 使用 LlamaIndex VectorStoreIndex 的 {model_name} RAG 系統測試\n", "=" * 50)

    try:
        rag = LlamaIndexRAG(model_name=model_name, embed_model_name=emodel_name)
    except ConnectionError as e:
        print(f"❌ 錯誤：{e}")
        return

    file_path = data_file 
    if not os.path.exists(file_path):
        print(f"❌ 錯誤：找不到文件 '{file_path}'。請確認文件路徑是否正確。")
        return
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    documents = content.split("=== 第")
    documents = documents[1:]
    documents = ["=== 第" + page.strip() for page in documents]

    if jb:
        segmented_docs = []
        for doc in documents:
            words = jieba.cut(doc)
            segmented_text = " ".join(words)
            segmented_docs.append(segmented_text)
    else:
        segmented_docs = documents

    print(f"共讀入 {len(segmented_docs)} 頁。")
    print(segmented_docs[0][:20])
    print(segmented_docs[-1][:20])
    rag.create_index_from_documents(segmented_docs, index_dir=idx_path )
    rag.print_vector_index_structure(max_nodes=100)

    print("\n🔍 請輸入查詢問題（輸入 quit 結束）\n", "=" * 60)
    while True:
        query = input("❓ 問題: ").strip()
        if query.lower() == "quit":
            print("\n✅ 結束測試。")
            break
        segmented_query = " ".join(jieba.cut(query)) if jb else query
        answer = rag.rag_query(segmented_query, 5)
        print(f"🤖 回答: {answer}\n", "-" * 50)

if __name__ == "__main__":
    main("114_碩士班招生簡章_分頁.txt", "app3_49_vector_storage",  # 49頁
         model_name="gpt-oss:20b", emodel_name="nomic-embed-text")

🤖 使用 LlamaIndex VectorStoreIndex 的 gpt-oss:20b RAG 系統測試
✅ Ollama 服務器 http://localhost:11434 可達。
共讀入 49 頁。
=== 第1 頁 ===
113 年 9
=== 第49 頁 ===
世新大學 1
✅ 成功載入現有索引於 'app3_49_vector_storage'。

📦 VectorStoreIndex 節點資訊:
共有 49 個節點。
-------------------------------------------------- 
[1] Node ID: bed55350-73af-48b9-8312-05a9650fcd08
來源檔案: N/A, 頁碼: N/A
 摘要: === 第1 頁 === 113 年 9 月 19 日本校   招生委員會會議通過                                       ...
-------------------------------------------------- 
[2] Node ID: 131bafb0-de4c-48ad-b83c-3be92cf429a9
來源檔案: N/A, 頁碼: N/A
 摘要: === 第2 頁 === 114 學 年 度 碩 士 班 招 生 重 要 日 程 表  項                目  日        期  簡章網路...
-------------------------------------------------- 
[3] Node ID: 38460529-1f0e-46cf-8d79-e4bf56690cd7
來源檔案: N/A, 頁碼: N/A
 摘要: === 第3 頁 === 報名流程圖    1.列印出之報名表請仔細核對並由本人親自簽名；若基本 資料有誤，如地址、性別等（非報考系所及選考科目） 請用紅筆直接...
-------------------------------------------------- 
[4] Node ID: 1d90ac8d-65b9-40a3-9689-b4f966af8d93
來源檔案: N/A, 頁碼: N/A
 摘要: === 第4 頁 === 世新大學 114

❓ 問題:  資訊管理學系碩士班 招生名額


🔍 正在查詢...


E:\RAG_class\Lib\site-packages\llama_index\core\base\embeddings\base.py:69: RuntimeWarning: invalid value encountered in scalar divide
  return product / norm


🤖 回答: 我不知道。

---
 📄 來源文件 (Source Nodes):
[1] Node ID: 48c83296-b793-4e04-b61b-f50723235c71
內容摘要: === 第48 頁 ...
 --------------------
[2] Node ID: 131bafb0-de4c-48ad-b83c-3be92cf429a9
內容摘要: === 第2 頁 =...
 --------------------
[3] Node ID: dff67404-5913-4959-bbe0-4f28d12ded91
內容摘要: === 第49 頁 ...
 --------------------
[4] Node ID: 1d90ac8d-65b9-40a3-9689-b4f966af8d93
內容摘要: === 第4 頁 =...
 --------------------
[5] Node ID: 3fd0c092-4071-4d58-a815-b22590a3f115
內容摘要: === 第5 頁 =...
 --------------------
🤖 回答: 我不知道。
 --------------------------------------------------
